# SatQuery AI — free hosting on Kaggle (demo endpoint)

Turns a Kaggle GPU session into a **public HTTPS endpoint** your frontend can
call, for free.

**How it works:** uvicorn serves `app.py` on `0.0.0.0:8000` inside the session,
and a Cloudflare quick tunnel exposes it at a `https://<random>.trycloudflare.com`
URL. No account, no token, no port forwarding.

## Read this before you start

| | |
|---|---|
| **Lifetime** | The URL dies when the session ends. Kaggle interactive sessions cap at ~12 h and idle out sooner. This is a **demo endpoint**, not production. |
| **Cold start** | ~5–10 min: pip installs plus the ~7 GB base-model download from Hugging Face. |
| **Quota** | Uses your Kaggle GPU allowance (~30 h/week). You have already spent several hours on training. |
| **Security** | The tunnel is **public and unauthenticated**. Anyone with the URL can call it. Do not put private data through it. Fine for a hackathon demo; not fine for anything real. |
| **Frontend** | Call the URL from the browser. It is a different origin, so you will need CORS — cell 7 adds it. |

## What you need attached

The two adapter zips from STEP 10. Either:

* **Attach your own v1 notebook's output as an Input** (Kaggle → your saved
  notebook → Output → Add as Input). That gives `satquery_adapters/` directly,
  no re-upload. **This is the easiest route.**
* Or upload `satquery_adapter_a_clean.zip` + `satquery_adapter_b_clean.zip` as a
  private Kaggle Dataset and attach that.

Cell 2 searches every mounted input for `adapter_model.safetensors`, so either
works. **Turn Internet ON** (Session options) or nothing will download.

In [ ]:
# 1 -- what are we running on?
import shutil, subprocess, sys
print("python :", sys.version.split()[0])
print("nvidia-smi:")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout.strip()
      or "  !! no GPU visible -- this backend needs CUDA")
print("disk free :", shutil.disk_usage("/kaggle/working").free // 2**20, "MB")
print("internet  :", subprocess.run(["python","-c",
      "import urllib.request;urllib.request.urlopen('https://huggingface.co',timeout=10);print('OK')"],
      capture_output=True,text=True).stdout.strip() or "!! UNREACHABLE -- turn Internet ON")

In [ ]:
# 2 -- find the adapters on the mounts
import json, shutil, zipfile
from pathlib import Path

DST = Path("/kaggle/working/satquery_backend/artifacts")
DST.mkdir(parents=True, exist_ok=True)

def find_adapters():
    """Return {a|b: dir} found on any mounted input, or from a zip."""
    out = {}
    hits = [p for p in Path("/kaggle/input").rglob("adapter_model.safetensors") if p.is_file()]
    for h in hits:
        d = h.parent
        letter = "a" if "adapter_a" in str(d) or "_a_" in d.name else (
                 "b" if "adapter_b" in str(d) else None)
        if letter and letter not in out:
            out[letter] = d
    for z in Path("/kaggle/input").rglob("satquery_adapter_*_clean.zip"):
        letter = "a" if "_a_" in z.name else "b"
        if letter in out:
            continue
        target = DST / f"adapter_{letter}"
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)
        # zips sometimes nest one level; flatten if so
        inner = [p for p in target.rglob("adapter_model.safetensors")]
        if inner and inner[0].parent != target:
            for f in inner[0].parent.iterdir():
                shutil.move(str(f), str(target / f.name))
        out[letter] = target
        print(f"  unzipped {z.name} -> {target}")
    return out

found = find_adapters()
for letter in ("a", "b"):
    d = found.get(letter)
    ok = d is not None and (Path(d) / "adapter_model.safetensors").is_file()
    print(f"  adapter_{letter}: {'FOUND ' + str(d) if ok else 'MISSING'}")
    if ok:
        for f in sorted(Path(d).iterdir()):
            print(f"      {f.name:<34} {f.stat().st_size/1e6:>8.2f} MB")
if len(found) < 2:
    raise SystemExit(
        "\n!! Could not find both adapters under /kaggle/input.\n"
        "!! Attach your saved training notebook's OUTPUT as an Input dataset\n"
        "!! (it contains satquery_adapters/), or upload the two\n"
        "!! satquery_adapter_*_clean.zip files as a private Kaggle Dataset,\n"
        "!! then re-run this cell.")
Path("/kaggle/working/_adapter_paths.json").write_text(
    json.dumps({k: str(v) for k, v in found.items()}))

In [ ]:
# 3 -- dependencies the Kaggle image does not already have
# torch / transformers / peft / bitsandbytes ship with the GPU image.
!pip install -q fastapi "uvicorn[standard]" python-multipart rasterio
# torchao breaks peft 0.20's is_torchao_available() -- see HANDOVER. Remove it.
!pip uninstall -y -q torchao 2>/dev/null || true
import importlib
for m in ("torch", "transformers", "peft", "bitsandbytes", "fastapi", "uvicorn", "rasterio"):
    try:
        mod = importlib.import_module(m)
        print(f"  {m:<14} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  {m:<14} !! {e}")
import torch
print("\n  cuda available :", torch.cuda.is_available())
print("  device count   :", torch.cuda.device_count())

In [ ]:
%%writefile /kaggle/working/satquery_backend/common.py
# common.py
import json
from pathlib import Path
import numpy as np
import rasterio
from PIL import Image
from rasterio.enums import Resampling

BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"


def _sar_to_uint8(arr):
    """
    Render a single SAR band to 8-bit greyscale.

    Four conventions appear in real SAR products and ALL FOUR must work:
      A  float linear power ~[0,1]          (calibrated SLC/GRD)  -> 10*log10
      B  linear power at an integer scale   (Sentinel-1 GRD and BigEarthNet-S1
                                             ship uint16 x10000)  -> 10*log10
      C  8-bit integer raster               (uint8 tif / preview that someone
                                             already rendered)    -> use directly
      D  float already in dB                (has negative values) -> use directly

    Normalisation then uses MEASURED 2/98 percentiles of the transformed array
    rather than a hardcoded [-25, 0] dB window. That is what makes the result
    calibration-scale invariant:

        log10(k * x) = log10(k) + log10(x)

    so any multiplicative calibration scale becomes a constant dB offset, and a
    percentile stretch cancels constant offsets exactly. The previous hardcoded
    window assumed float linear power in ~[0.003, 1]; every uint16 and every
    8-bit SAR input landed above 0 dB and clipped to a SOLID WHITE image, and
    pre-dB float input clipped to SOLID BLACK. The vision tower then received no
    signal at all for cross_modal, and the model narrated SAR content it had
    never seen.

    Non-finite and non-positive pixels (nodata) are excluded from the stretch
    and rendered black rather than being allowed to crush the histogram.
    """
    src = np.asarray(arr)
    if src.ndim == 3:                      # defensive: caller already took band 0
        src = src[..., 0]
    as_float = src.astype(np.float64)
    finite_mask = np.isfinite(as_float)
    if not finite_mask.any():
        return np.zeros(src.shape[:2], dtype=np.uint8)
    finite = as_float[finite_mask]

    is_int = np.issubdtype(src.dtype, np.integer)
    is_float = np.issubdtype(src.dtype, np.floating)

    if is_float and bool((finite < 0).any()):
        # D: already in dB. Logging again would be meaningless (log of a negative).
        vals = as_float
        invalid = ~finite_mask
    elif is_int and src.dtype.itemsize == 1:
        # C: 8-bit raster is an already-rendered product. Do not re-log it.
        vals = as_float
        invalid = ~finite_mask
    else:
        # A / B: linear power at some unknown scale.
        pos = finite[finite > 0]
        if pos.size == 0:
            vals = as_float                # no positive energy: nothing to log
            invalid = ~finite_mask
        else:
            floor = float(pos.min())       # data-relative epsilon, not a hardcoded 1e-5
            good = finite_mask & (as_float > 0)
            vals = 10.0 * np.log10(np.where(good, as_float, floor))
            invalid = ~good

    vals = np.where(invalid, np.nan, vals)
    valid = vals[np.isfinite(vals)]
    if valid.size == 0:
        return np.zeros(src.shape[:2], dtype=np.uint8)

    lo, hi = np.percentile(valid, [2, 98])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    out = np.clip((vals - lo) / (hi - lo), 0.0, 1.0) * 255.0
    out = np.nan_to_num(out, nan=0.0, posinf=255.0, neginf=0.0)
    return out.astype(np.uint8)


def load_image(spec, max_dim=512):
    path = Path(spec["path"])
    modality = spec.get("modality", "optical").lower()

    if path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        with Image.open(path) as img:
            img = img.convert("RGB")
            img.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
            arr = np.array(img, dtype=np.float32)
    else:
        with rasterio.open(path) as ds:
            bands = spec.get("bands", [1, 2, 3])
            max_b = ds.count
            bands = [b if b <= max_b else 1 for b in bands]

            scale = min(1.0, max_dim / max(ds.width, ds.height))
            h, w = max(1, round(ds.height * scale)), max(1, round(ds.width * scale))
            data = ds.read(bands, out_shape=(len(bands), h, w), resampling=Resampling.bilinear, masked=True)

            if hasattr(data, "filled"):
                arr = np.asarray(data.filled(0), dtype=np.float32)
            else:
                arr = np.nan_to_num(np.asarray(data, dtype=np.float32))

            if arr.ndim == 3:
                arr = np.transpose(arr, (1, 2, 0))

    if modality == "sar":
        if arr.ndim == 3:
            arr = arr[..., 0]
        sar_uint8 = _sar_to_uint8(arr)
        return Image.fromarray(np.stack([sar_uint8, sar_uint8, sar_uint8], axis=-1))

    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)

    rendered = []
    for c_idx in range(min(3, arr.shape[-1])):
        c = arr[..., c_idx]
        valid = c[np.isfinite(c)]
        lo, hi = np.percentile(valid, [2, 98]) if valid.size > 0 else (0.0, 255.0)
        if hi == lo: hi = lo + 1.0
        c_norm = np.clip((c - lo) / (hi - lo), 0, 1) * 255.0
        rendered.append(c_norm.astype(np.uint8))

    while len(rendered) < 3:
        rendered.append(rendered[0])

    return Image.fromarray(np.stack(rendered[:3], axis=-1))


def format_messages(row):
    content = []
    task = row["task"]

    for idx, img_spec in enumerate(row["images"]):
        mod = img_spec.get("modality", "optical").upper()
        date_str = f" ({img_spec['timestamp']})" if img_spec.get("timestamp") else ""

        if task == "cross_modal":
            label = f"Image {idx+1} [{mod}{date_str}]:"
        elif task == "change_vqa":
            label = f"Image {idx+1} [{'BEFORE' if idx==0 else 'AFTER'}{date_str}]:"
        else:
            label = f"Satellite Image [{mod}{date_str}]:"

        content.extend([{"type": "text", "text": label}, {"type": "image"}])

    system_prompt = (
        "You are SatQuery AI, an expert remote-sensing assistant. "
        "Examine the visual pixels of the image carefully and answer accurately. "
        "Do not invent water, buildings, or changes that are not clearly visible."
    )

    content.append({"type": "text", "text": f"Question: {row['query']}"})

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": content}
    ]


In [ ]:
%%writefile /kaggle/working/satquery_backend/app.py
# app.py
import os, time, tempfile, logging, threading
from pathlib import Path
from contextlib import asynccontextmanager
import torch
import rasterio
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from starlette.concurrency import run_in_threadpool
from peft import PeftModel
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from common import BASE_MODEL, load_image, format_messages


# ---------------------------------------------------------------------------
# [TASK 3] Real per-token sequence confidence.
# Pure helper over what generate() already returns; no new dependencies.
# ---------------------------------------------------------------------------
def compute_sequence_confidence(token_ids, scores, eos_token_id=None):
    """Mean softmax probability the model assigned to the token it actually chose.

    scores     : tuple, one (batch=1, vocab) logit tensor per decode step.
    token_ids  : (T,) tensor of the chosen token ids for those steps.

    Greedy decoding means token_ids[t] == argmax(scores[t]), so this is the mean
    top-1 probability over the generated content tokens. We stop at EOS so the
    terminator token does not inflate the score, and so trailing PADs are ignored.
    Returns a float in [0, 1]; 0.0 if nothing was generated.
    """
    prob_sum = 0.0
    counted = 0
    for step, logits in enumerate(scores):
        if step >= token_ids.shape[0]:
            break
        tok = int(token_ids[step])
        if eos_token_id is not None and tok == int(eos_token_id):
            break
        step_probs = torch.softmax(logits[0].float(), dim=-1)
        prob_sum += float(step_probs[tok])
        counted += 1
    if counted == 0:
        return 0.0
    return round(prob_sum / counted, 4)


class AgenticModelRuntime:
    def __init__(self, adapter_a_path, adapter_b_path):
        self.lock = threading.Lock()
        dtype = torch.float16
        print("[SatQuery AI] Initializing base model Qwen2.5-VL-3B-Instruct in 4-bit...")

        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
            llm_int8_skip_modules=["visual", "lm_head"]
        )

        # [TASK 1] Raised the spatial pixel budget.
        #   was: min_pixels=16*28*28, max_pixels=128*28*28
        #   now: min_pixels=64*28*28, max_pixels=256*28*28
        # 256*28*28 = 200704 px -> ~448x448 effective, 256 vision tokens/image.
        self.processor = AutoProcessor.from_pretrained(BASE_MODEL, min_pixels=64*28*28, max_pixels=256*28*28)

        base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            BASE_MODEL, quantization_config=quant, torch_dtype=dtype, device_map={"": 0}
        )

        self.model = base_model
        self.has_a = False
        self.has_b = False

        if os.path.exists(adapter_a_path):
            try:
                print(f"[SatQuery AI] Registering Adapter A from {adapter_a_path}...")
                self.model = PeftModel.from_pretrained(base_model, adapter_a_path, adapter_name="adapter_a")
                self.has_a = True
            except Exception as e:
                print(f"[SatQuery AI] Could not load Adapter A ({e}).")

        if os.path.exists(adapter_b_path) and self.has_a:
            try:
                print(f"[SatQuery AI] Registering Adapter B from {adapter_b_path}...")
                self.model.load_adapter(adapter_b_path, adapter_name="adapter_b")
                self.has_b = True
            except Exception as e:
                print(f"[SatQuery AI] Could not load Adapter B ({e}).")

        print("[SatQuery AI] Backend Engine Ready!")

    def predict(self, image_specs, task, query):
        torch.cuda.empty_cache()
        # [TASK 1 - follow-on, SEE HANDOVER FLAG F3] 384 -> 448 for image pairs.
        # The PIL pre-downscale in load_image() was binding BELOW the processor's
        # new 256*28*28 budget for pairs, so Task 1 would not have taken effect
        # on change_vqa / cross_modal at all. 448*448 = 200704 = exactly the new cap.
        max_dim = 448 if len(image_specs) == 2 else 512
        images = [load_image(spec, max_dim=max_dim) for spec in image_specs]

        with self.lock:
            if task == "change_vqa" and self.has_b:
                selected_adapter = "adapter_b"
                self.model.set_adapter("adapter_b")
            elif self.has_a:
                selected_adapter = "adapter_a"
                self.model.set_adapter("adapter_a")
            else:
                selected_adapter = "base_qwen_vlm"

            msgs = format_messages(row={"task": task, "query": query, "images": image_specs})
            prompt = self.processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

            inputs = self.processor(text=[prompt], images=images, return_tensors="pt").to("cuda:0")

            with torch.no_grad():
                # [TASK 3] return_dict_in_generate=True changes generate()'s return
                # value from a bare (batch, seq_len) token tensor into a
                # GenerateDecoderOnlyOutput object; the tokens move to .sequences.
                # output_scores=True makes it also collect .scores (one
                # (batch, vocab) logit tensor per step). Decoding behaviour itself
                # is unchanged: still greedy (do_sample=False), still 128 max tokens.
                gen_out = self.model.generate(
                    **inputs,
                    max_new_tokens=128,
                    do_sample=False,
                    return_dict_in_generate=True,
                    output_scores=True,
                )

            prompt_len = inputs["input_ids"].shape[1]
            new_tokens = gen_out.sequences[:, prompt_len:]
            answer = self.processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

            # [TASK 3] confidence from the model's own distribution, not a constant.
            tokenizer = getattr(self.processor, "tokenizer", None)
            eos_id = getattr(tokenizer, "eos_token_id", None) if tokenizer is not None else None
            confidence = compute_sequence_confidence(new_tokens[0], gen_out.scores, eos_token_id=eos_id)

        torch.cuda.empty_cache()
        return answer, selected_adapter, confidence


@asynccontextmanager
async def lifespan(app: FastAPI):
    adapter_a = os.environ.get("ADAPTER_A_PATH", "./artifacts/adapter_a")
    adapter_b = os.environ.get("ADAPTER_B_PATH", "./artifacts/adapter_b")
    app.state.runtime = AgenticModelRuntime(adapter_a, adapter_b)
    yield

app = FastAPI(title="SatQuery AI Agentic Backend", lifespan=lifespan)

@app.get("/health")
def health():
    return {"status": "ready", "base_model": BASE_MODEL}

@app.post("/analyze")
async def analyze(
    query: str = Form(...),
    dataset: str = Form("operational"),
    modalities: str = Form("optical"),
    timestamps: str = Form(""),
    bands: str = Form("1,2,3"),
    files: list[UploadFile] = File(...)
):
    try:
        if len(files) < 1 or len(files) > 2:
            raise HTTPException(400, "SatQuery AI accepts either 1 image or 2 images.")

        band_indices = [int(x.strip()) for x in bands.split(",")]
        modality_list = [m.strip().lower() for m in modalities.split(",")]

        # [TASK 2] Positional parse, NO empty-filtering (was `if t.strip()`).
        # Filtering shifted timestamps out of alignment with their file: ",2024-06-15"
        # used to land on file 0. Parsed positionally and padded, exactly like
        # bands / modalities, so index i always refers to file i.
        timestamp_list = [t.strip() for t in timestamps.split(",")]

        while len(modality_list) < len(files):
            modality_list.append("optical")
        while len(timestamp_list) < len(files):
            timestamp_list.append("")

        # [TASK 2 - SEE HANDOVER FLAG F4] Normalise "multispectral" -> "optical".
        # The problem statement's input scope is "optical/multispectral or SAR".
        # Without this, a valid multispectral pair would fall through to a 400.
        modality_list = ["sar" if m == "sar" else "optical" for m in modality_list]

        start_time = time.time()
        q_lower = query.lower()
        intent_basis = None

        # Guardrails & Routing
        if len(files) == 2:
            n_optical = sum(1 for m in modality_list if m == "optical")
            n_sar = sum(1 for m in modality_list if m == "sar")

            if n_optical >= 1 and n_sar >= 1:
                task = "cross_modal"
                # [TASK 4] Validate the fusion modality combination explicitly.
                if n_optical != 1 or n_sar != 1:
                    raise HTTPException(
                        400,
                        f"Cross-modal fusion requires exactly one optical and one SAR image "
                        f"(received {n_optical} optical, {n_sar} sar)."
                    )
                # [TASK 2] SAR-detection branch itself is unchanged (was already correct);
                # only the evidence string is new.
                intent_basis = (f"2 images, modalities={modality_list} -> "
                                "exactly one optical + one SAR co-registered pair")
            elif n_sar >= 1:
                # [TASK 4] was HTTP 422; Task 4 specifies HTTP 400 with this message.
                raise HTTPException(
                    400,
                    f"Cross-modal fusion requires exactly one optical and one SAR image "
                    f"(received {n_optical} optical, {n_sar} sar)."
                )
            else:
                # [TASK 2] change_vqa now REQUIRES two present, DIFFERENT timestamps.
                # No silent defaulting to change_vqa for an arbitrary 2-image upload.
                t1, t2 = timestamp_list[0], timestamp_list[1]
                if t1 and t2 and t1 != t2:
                    task = "change_vqa"
                    intent_basis = f"distinct timestamps provided ({t1} -> {t2})"

                    # Check if identical files were uploaded
                    f1_bytes = await files[0].read()
                    f2_bytes = await files[1].read()
                    await files[0].seek(0)
                    await files[1].seek(0)

                    if f1_bytes == f2_bytes:
                        return {
                            "task_intent": "change_vqa",
                            "query": query,
                            "answer": "No changes detected. The two input images are identical.",
                            "confidence": 1.0,
                            "confidence_source": "deterministic_byte_equality_guardrail",
                            "duration_seconds": round(time.time() - start_time, 3),
                            "auditable_execution_trace": [
                                {"tool": "guardrail_checker", "status": "identical_inputs_detected", "action": "bypassed_vlm"},
                                {"tool": "agentic_intent_classifier", "classified_task": "change_vqa", "basis": intent_basis},
                            ]
                        }
                else:
                    # [TASK 2] Ask the client to disambiguate instead of guessing.
                    reason = ("no timestamps supplied" if not (t1 and t2)
                              else f"both timestamps identical ({t1})")
                    raise HTTPException(
                        400,
                        "Ambiguous two-image intent: cannot distinguish a bi-temporal "
                        f"change pair from an unrelated image pair ({reason}). "
                        "To run change analysis, send `timestamps` with two DIFFERENT "
                        "acquisition dates (e.g. '2023-01-01,2024-06-15'). "
                        "To run cross-modal fusion, send `modalities=optical,sar`. "
                        "To analyse one scene, send a single image."
                    )

        else:
            if any(k in q_lower for k in ["describe", "caption", "scene description"]):
                task = "caption"
                intent_basis = "1 image, query matched caption keyword"
            else:
                task = "vqa"
                intent_basis = "1 image, no caption keyword -> default VQA"

        with tempfile.TemporaryDirectory() as temp_dir:
            image_specs = []
            image_metadata = []

            for idx, file in enumerate(files):
                ext = Path(file.filename or "").suffix.lower()
                temp_path = Path(temp_dir) / f"input_{idx}{ext}"
                content = await file.read()
                temp_path.write_bytes(content)

                ts = timestamp_list[idx] if idx < len(timestamp_list) else None
                ts = ts or None
                mod = modality_list[idx]

                spec = {"path": str(temp_path), "modality": mod, "bands": band_indices, "timestamp": ts}
                image_specs.append(spec)

                meta = {"filename": file.filename, "modality": mod, "timestamp": ts}
                image_metadata.append(meta)

            answer, chosen_adapter, confidence = await run_in_threadpool(
                app.state.runtime.predict,
                image_specs, task, query
            )

        duration = round(time.time() - start_time, 3)

        return {
            "task_intent": task,
            "query": query,
            "answer": answer,
            "confidence": confidence,
            "confidence_source": "mean_top1_token_probability",
            "duration_seconds": duration,
            "inputs": image_metadata,
            "auditable_execution_trace": [
                {"tool": "input_compatibility_checker", "status": "passed", "num_images": len(files), "modalities": modality_list, "timestamps": timestamp_list},
                {"tool": "agentic_intent_classifier", "classified_task": task, "basis": intent_basis},
                {"tool": "preprocessor", "rendering": "multisensor_sar_db_stretch" if "sar" in modality_list else "percentile_stretched_rgb"},
                {"tool": "specialist_registry", "selected_model": "Qwen2.5-VL-3B-Instruct", "active_adapter": chosen_adapter}
            ]
        }
    except HTTPException:
        raise
    except Exception as e:
        logging.exception("Analysis failed")
        raise HTTPException(500, str(e))


In [ ]:
# 6 -- launch uvicorn in the background and wait for /health
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

BK = Path("/kaggle/working/satquery_backend")
adapters = json.loads(Path("/kaggle/working/_adapter_paths.json").read_text())
env = dict(os.environ,
           ADAPTER_A_PATH=adapters["a"], ADAPTER_B_PATH=adapters["b"],
           PYTHONUNBUFFERED="1",
           CUDA_VISIBLE_DEVICES="0")     # one GPU: 4-bit + DataParallel is fatal
log = open("/kaggle/working/uvicorn.log", "wb")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(BK), stdout=log, stderr=subprocess.STDOUT, env=env)
print(f"uvicorn pid {{proc.pid}}; loading the base model takes a few minutes...")

deadline, ready = time.time() + 900, False
while time.time() < deadline:
    if proc.poll() is not None:
        print(Path("/kaggle/working/uvicorn.log").read_text()[-3000:])
        raise SystemExit(f"!! uvicorn exited with code {{proc.poll()}}")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as r:
            print("\n/health ->", r.read().decode()); ready = True; break
    except Exception:
        time.sleep(5)
        tail = Path("/kaggle/working/uvicorn.log").read_text().strip().splitlines()
        print(f"  ...waiting ({{int(deadline - time.time())}}s left)  {{tail[-1][:90] if tail else ''}}")
if not ready:
    print(Path("/kaggle/working/uvicorn.log").read_text()[-3000:])
    raise SystemExit("!! /health never came up. Paste the log above.")
Path("/kaggle/working/_uvicorn_pid").write_text(str(proc.pid))

In [ ]:
# 7 -- enable CORS so a browser frontend can call it
# Done at runtime rather than by editing app.py, so the tested backend file is
# unchanged. Wide-open origins: this is a throwaway demo tunnel.
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path
subprocess.run(["pip","install","-q","asgi-cors"], check=False)
pid = Path("/kaggle/working/_uvicorn_pid").read_text().strip()
subprocess.run(["kill", pid], check=False); time.sleep(3)
adapters = json.loads(Path("/kaggle/working/_adapter_paths.json").read_text())
env = dict(os.environ, ADAPTER_A_PATH=adapters["a"], ADAPTER_B_PATH=adapters["b"],
           PYTHONUNBUFFERED="1", CUDA_VISIBLE_DEVICES="0")
log = open("/kaggle/working/uvicorn.log", "wb")
proc = subprocess.Popen(
    [sys.executable, "-m", "asgi_cors", "--allow-origin", "*",
     "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/kaggle/working/satquery_backend", stdout=log, stderr=subprocess.STDOUT, env=env)
Path("/kaggle/working/_uvicorn_pid").write_text(str(proc.pid))
deadline = time.time() + 900
while time.time() < deadline:
    if proc.poll() is not None:
        print(Path("/kaggle/working/uvicorn.log").read_text()[-2500:])
        raise SystemExit("!! asgi-cors launch failed; paste the log above")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as r:
            print("/health ->", r.read().decode()); break
    except Exception:
        time.sleep(5)

In [ ]:
# 8 -- public HTTPS URL via a Cloudflare quick tunnel
import re, subprocess, time
from pathlib import Path
if not Path("/kaggle/working/cloudflared").exists():
    subprocess.run(["wget","-q","-O","/kaggle/working/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=False)
Path("/kaggle/working/cloudflared").chmod(0o755)
tlog = open("/kaggle/working/tunnel.log","wb")
tp = subprocess.Popen(["/kaggle/working/cloudflared","tunnel","--no-autoupdate",
                       "--url","http://127.0.0.1:8000"], stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    txt = Path("/kaggle/working/tunnel.log").read_text(errors="ignore")
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", txt)
    if m: url = m.group(0); break
    time.sleep(2)
if not url:
    print(Path("/kaggle/working/tunnel.log").read_text()[-2500:])
    raise SystemExit("!! tunnel did not produce a URL; paste the log above")
Path("/kaggle/working/PUBLIC_URL.txt").write_text(url)
print("=" * 72)
print("YOUR PUBLIC ENDPOINT")
print("=" * 72)
print(f"  health  : {url}/health")
print(f"  analyze : {url}/analyze        (POST, multipart)")
print(f"  docs    : {url}/docs           (Swagger UI, public!)")
print("=" * 72)
print("Give the frontend this base URL. It dies when the session ends.")
print("Saved to /kaggle/working/PUBLIC_URL.txt")

In [ ]:
# 9 -- prove the public URL actually works end to end
import json, subprocess, urllib.request
from pathlib import Path
url = Path("/kaggle/working/PUBLIC_URL.txt").read_text().strip()
with urllib.request.urlopen(f"{url}/health", timeout=30) as r:
    print("public /health ->", r.read().decode())
# find any raster to test with
cand = [p for p in Path("/kaggle/input").rglob("*")
        if p.suffix.lower() in {".tif",".tiff",".png",".jpg"} and p.is_file()]
if not cand:
    print("no image on the mounts to test with; skipping the /analyze call")
else:
    img = cand[0]
    print(f"\nPOSTing {{img.name}} ({{img.stat().st_size/1e3:.0f}} KB) to {{url}}/analyze ...")
    out = subprocess.run(["curl","-s","-X","POST",f"{{url}}/analyze",
        "-F","query=Describe the land-cover and major objects visible in this image.",
        "-F","modalities=optical","-F","bands=1,2,3","-F",f"files=@{{img}}"],
        capture_output=True, text=True)
    try:
        r = json.loads(out.stdout)
        print("\n  task       :", r.get("task"))
        print("  answer     :", r.get("answer"))
        print("  confidence :", r.get("confidence"))
        print("  adapter    :", r.get("adapter_used") or r.get("active_adapter"))
        print("  trace steps:", len(r.get("auditable_execution_trace", [])))
    except Exception:
        print("  raw:", out.stdout[:800], out.stderr[:400])

## Running it

Leave cells 6–9 running in the notebook. The endpoint lives as long as the
session does.

**Frontend:**

```js
const BASE = "https://<your-tunnel>.trycloudflare.com";
const fd = new FormData();
fd.append("query", "Has the built-up area changed between the two images?");
fd.append("modalities", "optical,optical");
fd.append("timestamps", "T1,T2");
fd.append("bands", "1,2,3");
fd.append("files", beforeFile);
fd.append("files", afterFile);
const r = await fetch(`${BASE}/analyze`, { method: "POST", body: fd });
const j = await r.json();   // answer, confidence, task, auditable_execution_trace
```

`GET {BASE}/docs` gives you Swagger UI on the public URL — useful for showing
judges the contract, but remember it is **public**.

## Stopping

```python
import subprocess
from pathlib import Path
subprocess.run(["pkill","-f","cloudflared"])
subprocess.run(["kill", Path("/kaggle/working/_uvicorn_pid").read_text().strip()])
```

## When the session dies

Re-run cells 2–9. The base model re-downloads (~7 GB) unless the HF cache
survives, so budget 5–10 minutes of cold start.

## If you need a URL that does not die

Every free option that persists is **CPU-only**, and this backend is built for
CUDA 4-bit. Hugging Face Spaces' free tier (2 vCPU / 16 GB) would run it, but
you would have to drop bitsandbytes and load the 3B model in fp32 — slow
(tens of seconds per query) and it needs `app.py` changes I have not made or
tested. For a hackathon demo the tunnel is the right trade. Say the word if you
want the CPU variant built and measured instead of guessed.